# ECABSD V3 — Kaggle Training Notebook

**Architecture:** 6-layer GATv2 + Cross-Attention + Global Fusion  
**Loss:** Combined Focal + Soft-Dice (directly optimises F1)  
**LR Schedule:** Linear Warmup → CosineAnnealingLR  
**Output:** `checkpoints/best_model_v3.pt` + `ecabsd_v3_results.zip`

---
**Before running:**
- ✅ Enable **GPU** (Settings → Accelerator → GPU T4 x2 or P100)
- ✅ Enable **Internet** (Settings → Internet → On)
- ✅ Attach your **ecabsd dataset** (must contain `processed/` and `splits.csv`)
- ✅ Run cells **top to bottom**

In [ ]:
# ============================================================
# CELL 1: Clone fresh repo
# ============================================================
import os, shutil

WORK    = '/kaggle/working'
REPO    = 'https://github.com/VigneshReddyKura/ecabsd.git'
WORKDIR = f'{WORK}/ecabsd'

os.chdir(WORK)

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
    print('[SETUP] Removed old repo.')

print('Cloning repo...')
!git clone {REPO} {WORKDIR}

# Verify key V3 files exist
for required in ['train_v3.py', 'evaluate_v3.py', 'models/ecabsd_v3_model.py', 'data/dataset.py']:
    path = os.path.join(WORKDIR, required)
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {required}: {status}')
    if not os.path.exists(path):
        raise RuntimeError(f'Clone failed — {required} not found. Push it to GitHub first.')

os.chdir(WORKDIR)
print('\nPWD:', os.getcwd())
print('✅ Repo ready.')

In [ ]:
# ============================================================
# CELL 2: GPU check + install dependencies
# ============================================================
import subprocess, sys, torch

print('=== Environment ===')
print('[GPU]    ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ NO GPU')
print('[CUDA]   ', torch.version.cuda)
print('[PyTorch]', torch.__version__)

def pip(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

print('\n[DEPS] Installing core packages...')
pip(['biopython', 'scikit-learn', 'pandas', 'pyyaml'])

print('[DEPS] Installing torch_geometric (2-3 min)...')
pip(['torch_geometric'])

# Try to install PyG extensions (may fail silently on some Kaggle envs — that is OK)
try:
    pip(['torch_scatter', 'torch_sparse', 'torch_cluster'])
    print('[DEPS] PyG extensions installed.')
except Exception as e:
    print(f'[DEPS] PyG extensions skipped (optional): {e}')

print('[DEPS] ✅ All dependencies installed.')

In [ ]:
# ============================================================
# CELL 3: Find dataset and copy into working dir
# ============================================================
import os, shutil, glob

WORKDIR = '/kaggle/working/ecabsd'
os.chdir(WORKDIR)

# Auto-scan /kaggle/input for a folder with processed/ + splits.csv
ds_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'processed' in dirs and 'splits.csv' in files:
        ds_dir = root
        break

if not ds_dir:
    # Show what is available so user can debug
    print('Available inputs:')
    for root, dirs, files in os.walk('/kaggle/input'):
        print(f'  {root}: dirs={dirs[:5]}, files={files[:5]}')
    raise RuntimeError('❌ Dataset not found! Attach the ecabsd dataset (must have processed/ and splits.csv).')

print(f'[DATA] Found dataset at: {ds_dir}')

# Copy processed graphs
if os.path.exists('data/processed'):
    shutil.rmtree('data/processed')
os.makedirs('data/processed', exist_ok=True)
shutil.copytree(os.path.join(ds_dir, 'processed'), 'data/processed', dirs_exist_ok=True)

# Copy splits
shutil.copy2(os.path.join(ds_dir, 'splits.csv'), 'data/splits.csv')

n_graphs = len(glob.glob('data/processed/*.pt'))
print(f'[DATA] Graphs copied : {n_graphs}')
print(f'[DATA] splits.csv    : {os.path.exists("data/splits.csv")}')

if n_graphs == 0:
    raise RuntimeError('❌ No .pt graph files found after copy!')

print('[DATA] ✅ Dataset ready.')

In [ ]:
# ============================================================
# CELL 4: Rebuild splits — zero overlap (leakage fix)
# ============================================================
import pandas as pd
from sklearn.model_selection import train_test_split

df          = pd.read_csv('data/splits.csv')
unique_pdbs = df['pdb_id'].unique()
print(f'[SPLITS] Total unique PDB complexes: {len(unique_pdbs)}')

train_ids, temp_ids = train_test_split(unique_pdbs, test_size=0.3, random_state=42)
val_ids,   test_ids = train_test_split(temp_ids,    test_size=0.5, random_state=42)

df.loc[df['pdb_id'].isin(train_ids), 'split'] = 'train'
df.loc[df['pdb_id'].isin(val_ids),   'split'] = 'val'
df.loc[df['pdb_id'].isin(test_ids),  'split'] = 'test'
df.to_csv('data/splits.csv', index=False)

vc = df['split'].value_counts()
print(f'  train residues : {vc.get("train", 0)}')
print(f'  val   residues : {vc.get("val",   0)}')
print(f'  test  residues : {vc.get("test",  0)}')

# Assert zero overlap at PDB level
t, v, e = set(train_ids), set(val_ids), set(test_ids)
assert len(t & v) == 0, f'Train-Val overlap: {len(t & v)}'
assert len(t & e) == 0, f'Train-Test overlap: {len(t & e)}'
assert len(v & e) == 0, f'Val-Test overlap: {len(v & e)}'
print('[LEAKAGE] ✅ Zero overlap confirmed.')

In [ ]:
# ============================================================
# CELL 5: Remove malformed graphs (wrong x / edge dims)
# ============================================================
import torch, glob, os, shutil

SRC = 'data/processed'
BAD = 'data/bad_graphs'
os.makedirs(BAD, exist_ok=True)

good = bad = 0
for f in glob.glob(SRC + '/*.pt'):
    try:
        g   = torch.load(f, map_location='cpu', weights_only=False)
        x_ok = hasattr(g, 'x') and g.x is not None and g.x.dim() == 2 and g.x.shape[1] == 33
        e_ok = hasattr(g, 'edge_attr') and g.edge_attr is not None and g.edge_attr.dim() == 2 and g.edge_attr.shape[1] == 5
        if x_ok and e_ok:
            good += 1
        else:
            bad += 1
            shutil.move(f, os.path.join(BAD, os.path.basename(f)))
    except Exception:
        bad += 1
        shutil.move(f, os.path.join(BAD, os.path.basename(f)))

print(f'Good graphs   : {good}')
print(f'Bad (quarantined) : {bad}')
print(f'Final graphs  : {len(glob.glob(SRC + "/*.pt"))}')

if good == 0:
    raise RuntimeError('❌ No valid graphs remaining! Check dataset quality.')

print('✅ Graph validation passed.')

In [ ]:
# ============================================================
# CELL 6: Write V3 config
# ============================================================
import yaml, os, pandas as pd

WORKDIR = '/kaggle/working/ecabsd'
os.chdir(WORKDIR)

with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# Data paths
cfg['data']['processed_dir'] = 'data/processed'
cfg['data']['splits_csv']    = 'data/splits.csv'

# V3 Model
cfg['model']['version']          = 'v3'
cfg['model']['esm_dim']          = 33
cfg['model']['edge_feature_dim'] = 5
cfg['model']['hidden_dim']       = 256
cfg['model']['num_gcn_layers']   = 6
cfg['model']['num_heads']        = 4
cfg['model']['dropout']          = 0.3

# Training
cfg['training']['epochs']                  = 120
cfg['training']['learning_rate']           = 0.0003
cfg['training']['weight_decay']            = 0.005
cfg['training']['batch_size']              = 8
cfg['training']['loss']                    = 'combined'
cfg['training']['focal_alpha']             = 0.75
cfg['training']['focal_gamma']             = 2.0
cfg['training']['dice_weight']             = 0.5
cfg['training']['warmup_epochs']           = 10
cfg['training']['early_stopping_patience'] = 60
cfg['training']['gradient_clip']           = 1.0
cfg['training']['chain_swap_prob']         = 0.5
cfg['training']['num_workers']             = 0
cfg['training']['seed']                    = 42

# Paths
cfg['paths']['checkpoints_dir'] = 'checkpoints'
cfg['paths']['logs_dir']        = 'logs'
cfg['paths']['results_dir']     = 'results'

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

df = pd.read_csv('data/splits.csv')
vc = df['split'].value_counts()

print('=== V3 Config Written ===')
print(f'  version        : {cfg["model"]["version"]}')
print(f'  hidden_dim     : {cfg["model"]["hidden_dim"]}')
print(f'  num_gcn_layers : {cfg["model"]["num_gcn_layers"]}')
print(f'  num_heads      : {cfg["model"]["num_heads"]}')
print(f'  epochs         : {cfg["training"]["epochs"]}')
print(f'  warmup         : {cfg["training"]["warmup_epochs"]}')
print(f'  train samples  : {vc.get("train", 0)}')
print(f'  val samples    : {vc.get("val",   0)}')
print(f'  test samples   : {vc.get("test",  0)}')
print('\n✅ All checks passed — safe to train V3!')

In [ ]:
# ============================================================
# CELL 7: TRAIN ECABSD V3
# ============================================================
import os, sys, subprocess, glob

WORKDIR = '/kaggle/working/ecabsd'
os.chdir(WORKDIR)

# Restore data module (dataset copy may have overwritten it)
print('[TRAIN] Restoring repo data module...')
subprocess.run(['git', 'checkout', '--', 'data/dataset.py', 'data/__init__.py'], cwd=WORKDIR)

if not os.path.exists('data/dataset.py'):
    raise FileNotFoundError('data/dataset.py missing after git restore!')

# Final sanity checks
print('[TRAIN] data/ contents :', os.listdir('data')[:20])
print('[TRAIN] Graphs          :', len(glob.glob('data/processed/*.pt')))
print('[TRAIN] splits.csv      :', os.path.exists('data/splits.csv'))
print('[TRAIN] train_v3.py     :', os.path.exists('train_v3.py'))
print('[TRAIN] ecabsd_v3_model :', os.path.exists('models/ecabsd_v3_model.py'))

if len(glob.glob('data/processed/*.pt')) == 0:
    raise RuntimeError('No .pt graphs in data/processed — cannot train.')

# Launch training (stdout streamed live)
print('\n' + '='*60)
print('  Launching ECABSD V3 Training')
print('='*60 + '\n')

process = subprocess.Popen(
    [sys.executable, '-u', 'train_v3.py'],
    cwd=WORKDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError('❌ Training failed — check error output above.')

print('\n✅ Training complete!')

In [ ]:
# ============================================================
# CELL 8: Evaluate V3 on test set
# ============================================================
import os, sys, subprocess

WORKDIR = '/kaggle/working/ecabsd'
os.chdir(WORKDIR)

print('[EVAL] Running evaluate_v3.py on test set...')
process = subprocess.Popen(
    [sys.executable, '-u', 'evaluate_v3.py'],
    cwd=WORKDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode != 0:
    print('⚠️  evaluate_v3.py returned non-zero. Check output above.')
else:
    print('\n✅ Evaluation complete.')

In [ ]:
# ============================================================
# CELL 9: Verify checkpoint + package for download
# ============================================================
import os, time, zipfile, glob

WORKDIR = '/kaggle/working/ecabsd'
os.chdir(WORKDIR)

ckpt = 'checkpoints/best_model_v3.pt'
if os.path.exists(ckpt):
    mt = os.path.getmtime(ckpt)
    sz = os.path.getsize(ckpt)
    print('✅ Checkpoint verified:')
    print(f'   best_model_v3.pt : {time.ctime(mt)}  ({sz // 1024} KB)')
else:
    print('❌ best_model_v3.pt NOT found — training may have failed.')

# List all checkpoints saved
all_ckpts = glob.glob('checkpoints/*.pt')
print(f'\nAll checkpoints saved: {[os.path.basename(c) for c in all_ckpts]}')

# Package everything for download
zip_path = '/kaggle/working/ecabsd_v3_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['checkpoints', 'logs', 'results']:
        for root, _, files in os.walk(folder):
            for f in files:
                zf.write(os.path.join(root, f))
    for root, _, files in os.walk('models'):
        for f in files:
            if f.endswith('.py'):
                zf.write(os.path.join(root, f))
    if os.path.exists('config.yaml'):
        zf.write('config.yaml')

print(f'\n✅ Download ready: {zip_path}')
print('Copy checkpoints/best_model_v3.pt into your local ecabsd/checkpoints/ folder.')